# weight-decay-decoupled composite — cx17: decoupled weight decay (AdamW): p *= (1 - lr*lam) then the Adam step

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `weight-decay-decoupled`, `inplace-param-update`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "weight-decay-decoupled"
DD_ATOM_IDS = ["weight-decay-decoupled", "inplace-param-update"]
DD_SUBTOPICS = ["Optimizer: decoupled weight decay (AdamW)", "PyTorch: In-place param update"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

AdamW (Loshchilov & Hutter 2017) fixed Adam's interaction with weight decay. The fix is subtle: **don't add `lam*theta` to the gradient**, instead **multiply theta by `(1 - lr*lam)` BEFORE the Adam step**. This 'decoupled' WD is what gives AdamW its name.

**Atom A — weight-decay-decoupled.** `p *= (1 - lr * lam)`. Note the LR appears here too — it's the same effective lr-scaled decay as you'd get from L2 in vanilla SGD, BUT it applies to `theta` directly rather than via the (whitened, second-moment-rescaled) Adam update.

**Atom B — inplace-param-update.** Both the WD shrink AND the Adam step are in-place on `p.data`. Done in-place because (i) the same tensor is reused across steps; (ii) it must be done outside the autograd graph (via `.data` or `t.no_grad()`).

**Anatomy of one AdamW step.**
```python
for p in params:
    g = p.grad
    # Atom A: decoupled WD FIRST (before the Adam adaptive update).
    p.data.mul_(1 - lr * weight_decay)
    # Adam moment updates.
    m.mul_(b1).add_(g, alpha=1 - b1)
    v.mul_(b2).addcmul_(g, g, value=1 - b2)
    step += 1
    m_hat = m / (1 - b1**step); v_hat = v / (1 - b2**step)
    # Atom B: in-place param update.
    p.data.addcdiv_(m_hat, v_hat.sqrt() + eps, value=-lr)
```

**Why both atoms together.** Run AdamW with `lam=0` → reduces exactly to Adam. Compare to Adam-with-L2-add (`g = g + lam*p`) — the trajectories DIFFER, because the L2-add gets rescaled by the Adam second-moment denom while the decoupled WD does not. This is the whole motivation for AdamW.

### Composite Exercise — decoupled weight decay (AdamW): p *= (1 - lr*lam) then the Adam step

**Atoms exercised together**: `weight-decay-decoupled`, `inplace-param-update`

Implement `cx17_adamw_step(params, lr, betas, eps, weight_decay, state)` for one AdamW optimizer step.

- `params`: list of `t.Tensor` with `.grad` populated.
- `betas`: `(b1, b2)` tuple, e.g. `(0.9, 0.999)`.
- `state`: dict `id(p) -> {'step': int, 'exp_avg': Tensor or None, 'exp_avg_sq': Tensor or None}`. Caller seeds with `{id(p): {'step': 0, 'exp_avg': None, 'exp_avg_sq': None}}`.

Per param:
1. Read `g = p.grad`.
2. **Decoupled WD on theta**: `p.data.mul_(1 - lr * weight_decay)` (in-place). Do this BEFORE the Adam moment updates.
3. Initialize `exp_avg` and `exp_avg_sq` to `t.zeros_like(p.data)` on the first step.
4. `state[id(p)]['step'] += 1`; let `s = state[id(p)]['step']`.
5. Adam moments: `m.mul_(b1).add_(g, alpha=1 - b1)`; `v.mul_(b2).addcmul_(g, g, value=1 - b2)`.
6. Bias-correct: `m_hat = m / (1 - b1**s)`; `v_hat = v / (1 - b2**s)`.
7. **In-place update**: `p.data.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)` (or equivalent in-place form on `p.data`).

The test cross-checks against `torch.optim.AdamW`.

In [ ]:
def cx17_adamw_step(params, lr, betas, eps, weight_decay, state):
    b1, b2 = betas
    for p in params:
        g = p.grad
        s = state[id(p)]
        # Atom A (weight-decay-decoupled): theta <- (1 - lr*lam) * theta, BEFORE Adam.
        if weight_decay != 0:
            p.data.mul_(1 - lr * weight_decay)
        # Lazy-init moments on first step.
        if s['exp_avg'] is None:
            s['exp_avg'] = t.zeros_like(p.data)
            s['exp_avg_sq'] = t.zeros_like(p.data)
        s['step'] += 1
        step = s['step']
        m = s['exp_avg']
        v = s['exp_avg_sq']
        # Adam EMA moments.
        m.mul_(b1).add_(g, alpha=1 - b1)
        v.mul_(b2).addcmul_(g, g, value=1 - b2)
        # Bias-correct.
        bc1 = 1 - b1 ** step
        bc2 = 1 - b2 ** step
        m_hat = m / bc1
        v_hat = v / bc2
        # Atom B (inplace-param-update): one in-place addcdiv on p.data.
        denom = v_hat.sqrt().add_(eps)
        p.data.addcdiv_(m_hat, denom, value=-lr)


<details><summary>Show solution — cx17</summary>

```python
def cx17_adamw_step(params, lr, betas, eps, weight_decay, state):
    b1, b2 = betas
    for p in params:
        g = p.grad
        s = state[id(p)]
        # Atom A (weight-decay-decoupled): theta <- (1 - lr*lam) * theta, BEFORE Adam.
        if weight_decay != 0:
            p.data.mul_(1 - lr * weight_decay)
        # Lazy-init moments on first step.
        if s['exp_avg'] is None:
            s['exp_avg'] = t.zeros_like(p.data)
            s['exp_avg_sq'] = t.zeros_like(p.data)
        s['step'] += 1
        step = s['step']
        m = s['exp_avg']
        v = s['exp_avg_sq']
        # Adam EMA moments.
        m.mul_(b1).add_(g, alpha=1 - b1)
        v.mul_(b2).addcmul_(g, g, value=1 - b2)
        # Bias-correct.
        bc1 = 1 - b1 ** step
        bc2 = 1 - b2 ** step
        m_hat = m / bc1
        v_hat = v / bc2
        # Atom B (inplace-param-update): one in-place addcdiv on p.data.
        denom = v_hat.sqrt().add_(eps)
        p.data.addcdiv_(m_hat, denom, value=-lr)
```

Two failure modes the cross-check catches: (1) Putting `p.data.mul_(1 - lr*lam)` AFTER the Adam step changes which `theta` value the decay applies to (post-step vs pre-step), and diverges from `torch.optim.AdamW` by one step. (2) Adding `lam*p` to `g` (the L2-add form) gives a NUMERICALLY DIFFERENT trajectory because the Adam denom (`v_hat.sqrt() + eps`) rescales the L2 contribution — exactly the bug Loshchilov & Hutter pointed out. The in-place `addcdiv_` is one fused kernel doing `p += -lr * m_hat / denom`.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx17'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx17',
        'subtopics': ["Optimizer: decoupled weight decay (AdamW)", "PyTorch: In-place param update"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()